# Step1: QC + Preprocessing v2

Canonical real-project notebook for scLucid `QC + preprocess`.

This version is intentionally narrower than the older project notebook:

- QC uses the canonical reviewer-first entrypoint `scl.qc.run_qc()`.
- Preprocessing uses `scl.pp.run_iterative_preprocessing()` instead of a long notebook-local assembly.
- Validation is explicit: QC contract, preprocess contract, and lightweight comparative-readiness scaffold are all checked before saving the final object.
- The notebook is meant for **real project Step1 execution with human review**, not for blind fully automatic filtering.


In [ ]:
import json
import warnings
from pathlib import Path

import pandas as pd
import scanpy as sc
import scLucid as scl

warnings.filterwarnings("ignore")
scl.setup_logging("INFO")
scl.set_figure_params(dpi=150, dpi_save=300, figsize=(6, 5), style="seaborn-v0_8")

display(pd.Series({"scLucid_version": getattr(scl, "__version__", "unknown")}))


In [ ]:
# ------------------------------------------------------------------
# Project configuration
# ------------------------------------------------------------------
PROJECT_NAME = "project_step1_v2"
DATASET_ROLE = "real_project"

RAW_DIR = Path("data/raw/0-RAW")
STEP0_FILE = Path("data/processed/Step0-combined_raw_data.h5ad")
STEP1_FILE = Path("data/processed/Step1-sce_cleaned_v2.h5ad")
STEP2_FILE = Path("data/processed/Step2-sce_preprocessed_v2.h5ad")
RESULTS_DIR = Path("results") / PROJECT_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ALL_SAMPLES = []
SAMPLE_KEY = "sampleID"
GROUP_KEY = "group"
MODEL_KEY = None
GROUP_DICT = {}
MODEL_DICT = {}

SPECIES = "human"
TISSUE = "brain"
TISSUE_TYPE = "normal_tissue"  # use tumor_tissue for tumor projects
DATASET_TYPE = "multi_sample"
CANCER_TYPE = None

# QC policy
DOUBLET_METHOD = "scrublet"
THRESHOLD_MODE = "hierarchical"
QC_FINAL_FILTER_POLICY = "decision_remove"  # decision_remove / legacy / none
RUN_QUICK_QC_REVIEW = True
QC_DECISION_POLICY = "screening"
MANUAL_GUARDRAILS = {
    "min_genes": 200,
    "max_genes": 8000,
    "min_counts": None,
    "max_counts": 80000,
    "pc_mt": 15,
    "nmads": 5.0,
}

# Preprocess policy
N_TOP_GENES = 2500
N_PCS = 30
N_NEIGHBORS = 15
RUN_REGRESSION = False
REGRESS_VARS = ["total_counts", "pct_counts_mt"]
PREPROCESS_BATCH_KEY = SAMPLE_KEY
INTEGRATION_METHOD = "harmony"
INTEGRATION_POLICY = "auto_review"  # auto_review / force / off
RUN_HVG_STABILITY = False
RUN_DIAGNOSTIC_EMBEDDING = True
RUN_RECOMMENDATION_REVIEW = True

display(pd.Series({
    "project_name": PROJECT_NAME,
    "tissue": TISSUE,
    "tissue_type": TISSUE_TYPE,
    "sample_key": SAMPLE_KEY,
    "group_key": GROUP_KEY,
    "integration_policy": INTEGRATION_POLICY,
})).to_frame("value")


In [ ]:
# ------------------------------------------------------------------
# Data loading
# ------------------------------------------------------------------
metadata_dicts = scl.utils.build_metadata_dicts(
    samples=ALL_SAMPLES,
    group_dict=GROUP_DICT,
    batch_dict=MODEL_DICT,
    group_key=GROUP_KEY,
    batch_key=MODEL_KEY,
)

if STEP0_FILE.exists():
    adata = sc.read_h5ad(str(STEP0_FILE))
elif RAW_DIR.exists() and ALL_SAMPLES:
    adata = scl.utils.load_10x_data(
        samples=ALL_SAMPLES,
        base_dir=str(RAW_DIR),
        metadata_dicts=metadata_dicts,
        output_file=str(STEP0_FILE),
    )
else:
    raise FileNotFoundError(
        "Provide Step0-combined_raw_data.h5ad or configure RAW_DIR + ALL_SAMPLES."
    )

if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()
else:
    adata.X = adata.layers["counts"].copy()

if SAMPLE_KEY not in adata.obs.columns:
    adata.obs[SAMPLE_KEY] = "sample_1"
if GROUP_KEY and GROUP_KEY not in adata.obs.columns and GROUP_DICT:
    adata.obs[GROUP_KEY] = adata.obs[SAMPLE_KEY].map(GROUP_DICT).fillna("Unknown")
if MODEL_KEY and MODEL_KEY not in adata.obs.columns and MODEL_DICT:
    adata.obs[MODEL_KEY] = adata.obs[SAMPLE_KEY].map(MODEL_DICT).fillna("Unknown")

input_shape = {"n_cells": int(adata.n_obs), "n_genes": int(adata.n_vars)}
print(f"Loaded {adata.n_obs:,} cells x {adata.n_vars:,} genes")
display(adata)


In [ ]:
# ------------------------------------------------------------------
# Reviewer-first QC
# ------------------------------------------------------------------
qc_dir = RESULTS_DIR / "qc"
qc_dir.mkdir(parents=True, exist_ok=True)

qc_config = scl.qc.QCWorkflowConfig(
    sample_key=SAMPLE_KEY,
    species=SPECIES,
    tissue=TISSUE,
    tissue_type=TISSUE_TYPE,
    threshold_mode=THRESHOLD_MODE,
    use_recommendations=True,
    run_decision_engine=True,
    qc_decision_policy=QC_DECISION_POLICY,
    save_dir=str(qc_dir),
    use_parallel=False,
    n_jobs=1,
)
qc_config.metrics_reporting_config.plot_violin = False
qc_config.metrics_reporting_config.plot_scatter = False
qc_config.metrics_reporting_config.plot_top_genes = False
qc_config.metrics_reporting_config.show_plots = False
qc_config.marking_config.plot_outliers = False
qc_config.marking_config.show_plots = False
qc_config.doublet_config.method = DOUBLET_METHOD
qc_config.doublet_config.plot_summary = False
qc_config.doublet_config.plot_bar = False
qc_config.doublet_config.plot_scatter = False
qc_config.doublet_config.plot_upset = False
qc_config.doublet_config.show_plots = False
qc_config.marking_config.thresholds = scl.qc.QCThresholds(**MANUAL_GUARDRAILS)

if SAMPLE_KEY in adata.obs.columns:
    qc_config.doublet_config.expected_doublet_rate = scl.qc.generate_doublet_rates(
        adata,
        sample_key=SAMPLE_KEY,
    )

adata_qc = scl.qc.run_qc(
    adata,
    config=qc_config,
    tissue_type=TISSUE_TYPE,
    final_filter_policy=QC_FINAL_FILTER_POLICY,
    run_quick_review=RUN_QUICK_QC_REVIEW,
    show_progress=True,
    overwrite=True,
)

qc_review = adata_qc.uns["sclucid"]["qc"]["review_summary"]
qc_validation = scl.qc.validate_qc_module_completeness(adata_qc)
qc_compact = scl.qc.summarize_qc_review_summary(qc_review)
display(pd.Series(qc_compact))
display(pd.Series({
    "qc_valid": qc_validation["valid"],
    "qc_status": qc_validation["status"],
    "qc_readiness": qc_compact.get("readiness_status"),
    "qc_maturity": qc_compact.get("maturity_status"),
    "retained_cells": int(adata_qc.n_obs),
})).to_frame("value")

qc_save_path = scl.ut.write_h5ad_safe(
    adata_qc,
    STEP1_FILE,
    compression="gzip",
    sanitize_uns=True,
    atomic=True,
)
print(f"Saved QC object: {qc_save_path}")


In [ ]:
# ------------------------------------------------------------------
# Optional parameter recommendation checkpoint
# ------------------------------------------------------------------
recommendation_summary = {}
if RUN_RECOMMENDATION_REVIEW:
    recommendation_dir = RESULTS_DIR / "recommendation_review"
    recommendation_dir.mkdir(parents=True, exist_ok=True)
    recommendation_bundle = scl.recommendation.recommend_analysis_parameters(
        adata_qc,
        dataset_type=DATASET_TYPE,
        batch_key=PREPROCESS_BATCH_KEY if PREPROCESS_BATCH_KEY in adata_qc.obs.columns else None,
        tissue_type=TISSUE_TYPE,
        tissue=TISSUE,
        species=SPECIES,
        cancer_type=CANCER_TYPE,
        plot=False,
        save_dir=recommendation_dir,
    )
    recommendation_summary = recommendation_bundle.to_dict()
    (recommendation_dir / "workflow_recommendations.json").write_text(
        json.dumps(recommendation_summary, indent=2, default=str),
        encoding="utf-8",
    )
    print(f"Recommendation sidecar: {recommendation_dir / 'workflow_recommendations.json'}")
else:
    print("Recommendation checkpoint skipped.")


In [ ]:
# ------------------------------------------------------------------
# Canonical iterative preprocessing
# ------------------------------------------------------------------
pp_dir = RESULTS_DIR / "preprocess"
pp_dir.mkdir(parents=True, exist_ok=True)

preprocess_config = scl.pp.WorkflowConfig.quick(
    n_top_genes=N_TOP_GENES,
    run_regression=RUN_REGRESSION,
    run_integration=bool(INTEGRATION_METHOD and PREPROCESS_BATCH_KEY in adata_qc.obs.columns),
    save_dir=str(pp_dir),
    n_jobs=1,
)
preprocess_config.plot = False
preprocess_config.normalization.plot = False
preprocess_config.normalization.report = False
preprocess_config.hvg.plot = False
preprocess_config.hvg.report = False
preprocess_config.scaling.plot = False
preprocess_config.scaling.report = False
preprocess_config.graph.plot = False
preprocess_config.graph.report = False
preprocess_config.graph.n_pcs = N_PCS
preprocess_config.graph.n_neighbors = N_NEIGHBORS
preprocess_config.integration.plot = False
preprocess_config.integration.report = False
preprocess_config.integration.method = INTEGRATION_METHOD if PREPROCESS_BATCH_KEY in adata_qc.obs.columns else None
preprocess_config.integration.batch_key = PREPROCESS_BATCH_KEY if PREPROCESS_BATCH_KEY in adata_qc.obs.columns else None
preprocess_config.integration.use_rep = "X_pca"
preprocess_config.integration.evaluate = False
preprocess_config.scaling.vars_to_regress = [v for v in REGRESS_VARS if v in adata_qc.obs.columns]

biology_columns = [col for col in [GROUP_KEY, MODEL_KEY] if col and col in adata_qc.obs.columns]
condition_key = GROUP_KEY if GROUP_KEY in adata_qc.obs.columns else None

adata_pp = scl.pp.run_iterative_preprocessing(
    adata_qc,
    config=preprocess_config,
    save_dir=str(pp_dir),
    tissue_type=TISSUE_TYPE,
    sample_key=SAMPLE_KEY,
    biology_columns=biology_columns,
    condition_key=condition_key,
    integration_policy=INTEGRATION_POLICY,
    run_hvg_stability=RUN_HVG_STABILITY,
    run_diagnostic_embedding=RUN_DIAGNOSTIC_EMBEDDING,
    optimize_final_graph=True,
    diagnostic_umap_key="X_umap_diagnostic",
    final_umap_key="X_umap",
    show_progress=True,
    inplace=False,
    keep_intermediate_layers=True,
)

if "highly_variable" not in adata_pp.var.columns:
    fallback_hvg = next((c for c in adata_pp.var.columns if str(c).startswith("highly_variable")), None)
    if fallback_hvg is not None:
        adata_pp.var["highly_variable"] = adata_pp.var[fallback_hvg].astype(bool)

preprocess_review = adata_pp.uns["sclucid"]["preprocess"]["review_summary"]
pp_validation = scl.pp.validate_preprocess_module_completeness(adata_pp)
pp_compact = scl.pp.summarize_preprocess_review_summary(preprocess_review)
display(pd.Series(pp_compact))
display(pd.Series({
    "preprocess_valid": pp_validation["valid"],
    "preprocess_status": pp_validation["status"],
    "preprocess_readiness": pp_compact.get("readiness_status"),
    "n_hvg_selected": pp_compact.get("n_hvg_selected"),
    "umap_present": "X_umap" in adata_pp.obsm,
})).to_frame("value")


In [ ]:
# ------------------------------------------------------------------
# Validation scaffold + save final object
# ------------------------------------------------------------------
validation = scl.ut.build_qc_preprocess_validation(
    adata_pp,
    run_manifest={
        "workflow": "step1_qc_preprocess_v2",
        "input_shape": input_shape,
        "retention_fraction": float(adata_qc.n_obs / max(input_shape["n_cells"], 1)),
        "dataset_role": DATASET_ROLE,
    },
    dataset_role=DATASET_ROLE,
    workflow_name=f"{PROJECT_NAME}_step1_v2",
)
validation_artifacts = scl.ut.write_validation_outputs(validation, RESULTS_DIR / "validation")
display(pd.DataFrame(validation["compact_validation_table"]))
display(pd.Series({
    "ready_for_comparative_validation": validation["ready_for_comparative_validation"],
    "readiness_status": validation["readiness_status"],
    "validation_json": validation_artifacts["json"],
    "validation_table": validation_artifacts["table_csv"],
})).to_frame("value")

if "sclucid" in adata_pp.uns:
    adata_pp.uns["sclucid"] = scl.ut.sanitize_for_hdf5(adata_pp.uns["sclucid"])

final_path = scl.ut.write_h5ad_safe(
    adata_pp,
    STEP2_FILE,
    compression="gzip",
    sanitize_uns=False,
    atomic=True,
)
print(f"Saved final preprocess object: {final_path}")
print("Review before downstream analysis:")
print(f"  QC review summary: {qc_dir / 'qc_review_summary.json'}")
print(f"  Preprocess review summary: {pp_dir / 'preprocess_review_summary.json'}")
print(f"  Validation JSON: {validation_artifacts['json']}")


## Review checklist before Step2/analysis

- If `qc_validation["valid"]` is false, fix QC contract or review warnings before proceeding.
- If `pp_validation["valid"]` is false, do not trust downstream clustering/annotation yet.
- If `validation["ready_for_comparative_validation"]` is false, treat the output as a draft object and inspect the blocking failures.
- In stress-rich, tumor, BBB, or fragile-cell datasets, manually inspect high-MT and sample-specific retention patterns before accepting irreversible filtering.
- When integration ran, compare `X_umap_diagnostic` with final `X_umap` before using integrated coordinates for biological claims.
